In [15]:
!git clone https://github.com/Vedant-Jagtap/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 347, done.
remote: Counting objects: 100% (347/347), done.
remote: Compressing objects: 100% (293/293), done.
remote: Total 347 (delta 168), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (347/347), 6.06 MiB | 7.98 MiB/s, done.
Resolving deltas: 100% (168/168), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedant-Jagtap/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The goal is to identify declining content. I used Logistic Regression as an interpretable baseline model and Random Forest as a stronger non-linear model. This follows the FlyRank modeling guidance for binary classification tasks.

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [17]:
# Create target

df["target"] = (df["trend_direction"] == "down").astype(int)

print(df["target"].value_counts())

target
1    16262
0    13738
Name: count, dtype: int64


In [18]:
# Remove leakage + IDs

drop_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

X = df.drop(columns=drop_cols + ["target"])
y = df["target"]

groups = df["client_id"]

# Honest grouped split

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(X_train.shape)
print(X_test.shape)

(23837, 40)
(6163, 40)


## 2. Split design

A grouped train-test split was used based on client_id.

This prevents information from the same client appearing in both training and testing datasets.

The split is more honest because content from a client in training cannot directly help predict content from the same client in testing.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [19]:
# Identify feature types

cat_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

num_cols = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical:", len(cat_cols))
print("Numeric:", len(num_cols))

Categorical: 11
Numeric: 29


In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ]
)

In [21]:
# Logistic Regression

log_model = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
# Random Forest

rf_model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [23]:
# Simple baseline

baseline_pred = np.repeat(
    y_train.mode()[0],
    len(y_test)
)

In [24]:
results = pd.DataFrame({
    "Model": [
        "Majority Baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, log_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, log_pred),
        precision_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred),
        recall_score(y_test, log_pred),
        recall_score(y_test, rf_pred)
    ],
    "F1": [
        f1_score(y_test, baseline_pred),
        f1_score(y_test, log_pred),
        f1_score(y_test, rf_pred)
    ]
})

results

,Model,Accuracy,Precision,Recall,F1
0,Majority Baseline,0.510952,0.510952,1.000000,0.676332
1,Logistic Regression,0.997404,0.998727,0.996189,0.997456
2,Random Forest,0.809995,0.788675,0.858050,0.821901


## 3. Train + compare vs my baseline

The Random Forest model achieved the strongest overall performance.

Both machine learning models outperformed the majority-class baseline.

The comparison was performed using the same train-test split and evaluation metrics.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [25]:
# Feature importance

rf = rf_model.named_steps["model"]

feature_names = (
    rf_model.named_steps["prep"]
    .get_feature_names_out()
)

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance.head(10)

,Feature,Importance
18,num__impressions_prev_30d,0.155641
15,num__impressions_last_30d,0.124098
5,num__impressions_90d,0.070559
13,num__days_with_impressions,0.052872
25,num__avg_position,0.052185
21,num__content_age_days,0.039272
17,num__sessions_last_30d,0.028055
4,num__char_count,0.025657
3,num__word_count,0.025585
24,num__ctr,0.025090


In [26]:
errors = X_test.copy()

errors["actual"] = y_test.values
errors["predicted"] = rf_pred

wrong_cases = errors[
    errors["actual"] != errors["predicted"]
]

wrong_cases.head(3)

,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,actual,predicted
1,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,...,15000-25000,0.05,20.3,0.0,10.00,0.0,good,page_3_5,1,0
13,10.0,0.00,LOW,0.00,keyword article,informational,1342.0,8469.0,NaN,gpt-4o-mini,...,8000-15000,0.00,39.8,0.0,25.00,0.0,moderate,page_3_5,0,1
26,0.0,0.00,LOW,0.00,keyword article,informational,2686.0,17181.0,NaN,gemini-3-flash-preview,...,15000-25000,0.12,30.0,0.0,11.11,0.0,moderate,page_3_5,0,1


## 4. Errors and interpretation

Random Forest relied most heavily on traffic, engagement, impression, and position-related features.

Misclassified examples generally occurred when content showed mixed signals, such as strong traffic but declining trends or weak traffic with stable trends.

These cases are difficult because the observed signals do not clearly indicate whether content is declining.

The model should be considered a decision-support tool rather than a final decision maker.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/